# Threshold Optimization — Finding the EV Sweet Spot

The model has 78.6% WR but only ~0.78 trades/day. Can we loosen the filter to get more trades
while keeping positive EV?

| Analysis | What it answers |
|----------|----------------|
| 1. Threshold sweep | WR, EV, trade count, total PnL across |Q50| thresholds |
| 2. Optimal threshold | Which threshold maximizes total PnL (or daily PnL) |
| 3. No-filter baseline | What happens if we trade every hour (|Q50| > 0) |
| 4. Per-pair breakdown | Some pairs might work at lower thresholds than others |
| 5. Direction-only filter | Trade when Q50 sign is clear, ignore magnitude |
| 6. Quantile spread filter | Use Q75-Q25 as uncertainty — trade when IQR is tight |

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

DARK  = '#080c14'
BLUE  = '#4fc3f7'
GOLD  = '#ffd700'
RED   = '#ef5350'
GREEN = '#66bb6a'

DATA_DIR   = Path('../backend/data/features_2')
MODELS_DIR = Path('../backend/models_5')
TRAIN_END  = '2024-06-30'
AVG_SPREAD = 0.00028

QUANTILES      = [0.10, 0.25, 0.50, 0.75, 0.90]
QUANTILE_NAMES = ['Q10', 'Q25', 'Q50', 'Q75', 'Q90']
TARGET_COL     = 'label_1H'

# -- Load data --
df = pd.read_parquet(DATA_DIR / 'all_pairs_microstructure.parquet')
df.index = pd.to_datetime(df.index)

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]

# -- Load models --
models = {}
for q_name in QUANTILE_NAMES:
    q_int = int(float(q_name[1:]))
    bundle = joblib.load(MODELS_DIR / f'model_1H_Q{q_int}.joblib')
    models[q_name] = bundle['model']

# -- Test set --
df_test = df[df.index > TRAIN_END].copy()
X_test = df_test[feature_cols].ffill().fillna(0)
y_test = df_test[TARGET_COL].values
pairs  = df_test['pair'].values

# -- Predictions --
q_preds = {}
for q_name in QUANTILE_NAMES:
    q_preds[q_name] = models[q_name].predict(X_test)

q50 = q_preds['Q50']
q10 = q_preds['Q10']
q25 = q_preds['Q25']
q75 = q_preds['Q75']
q90 = q_preds['Q90']
pred_dir = np.sign(q50)
abs_q50  = np.abs(q50)

# Exclude NaN labels
valid = ~np.isnan(y_test)

test_days = (df_test.index.max() - df_test.index.min()).days
n_pairs = df_test['pair'].nunique()
print(f'Test: {valid.sum():,} valid hours | {test_days} days | {n_pairs} pairs')
print(f'Predictions ready: Q10..Q90')

---
## 1. Threshold Sweep — |Q50| filter

Sweep from 0 (trade every hour) to 3x spread. For each threshold: trades/day, WR, EV/trade, total PnL.

In [ ]:
thresholds = np.concatenate([
    [0],
    np.arange(0.00002, 0.00020, 0.00002),
    np.arange(0.00020, 0.00100, 0.00005),
])

results = []
for thr in thresholds:
    mask = (abs_q50 > thr) & valid
    n = mask.sum()
    if n < 10:
        continue
    
    pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
    wins = (pred_dir[mask] == np.sign(y_test[mask]))
    
    results.append({
        'threshold': thr,
        'thr_vs_spread': thr / AVG_SPREAD,
        'n_trades': n,
        'trades_per_day': n / test_days,
        'win_rate': wins.mean(),
        'ev_per_trade': pnl.mean(),
        'total_pnl': pnl.sum(),
        'daily_pnl': pnl.sum() / test_days,
        'sharpe': pnl.mean() / (pnl.std() + 1e-10) * np.sqrt(252 * 24),
    })

res_df = pd.DataFrame(results)
print(f'{"Thr":>8} {"xSpread":>8} {"Trades":>7} {"Tr/day":>7} {"WR":>7} {"EV/trade":>10} {"TotalPnL":>10} {"DailyPnL":>10} {"Sharpe":>8}')
print('-' * 90)
for _, r in res_df.iterrows():
    print(f'{r["threshold"]:>8.5f} {r["thr_vs_spread"]:>7.1f}x {r["n_trades"]:>7.0f} {r["trades_per_day"]:>7.1f} {r["win_rate"]:>6.1%} {r["ev_per_trade"]:>10.6f} {r["total_pnl"]:>10.4f} {r["daily_pnl"]:>10.6f} {r["sharpe"]:>8.1f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(DARK)

plots = [
    ('trades_per_day', 'Trades / Day', BLUE),
    ('win_rate', 'Win Rate', GREEN),
    ('ev_per_trade', 'EV per Trade', GOLD),
    ('total_pnl', 'Total PnL (log-return)', GOLD),
    ('daily_pnl', 'Daily PnL', GREEN),
    ('sharpe', 'Annualized Sharpe', BLUE),
]

x = res_df['thr_vs_spread'].values

for ax, (col, title, color) in zip(axes.flat, plots):
    ax.set_facecolor(DARK)
    ax.plot(x, res_df[col].values, 'o-', color=color, markersize=4, linewidth=1.5)
    
    # Mark the current threshold (1x spread)
    idx_1x = (res_df['thr_vs_spread'] - 1.0).abs().idxmin()
    ax.axvline(1.0, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
    
    if col == 'win_rate':
        ax.axhline(0.5, color=RED, linewidth=0.5, linestyle='--', alpha=0.5)
    if col in ('ev_per_trade', 'total_pnl', 'daily_pnl'):
        ax.axhline(0, color=RED, linewidth=0.5, linestyle='--', alpha=0.5)
    
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Threshold (x spread)', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Threshold Sweep — |Q50| Filter', color='white', fontsize=14)
plt.tight_layout()
plt.show()

# Find optimal thresholds
best_total = res_df.loc[res_df['total_pnl'].idxmax()]
best_daily = res_df.loc[res_df['daily_pnl'].idxmax()]
best_sharpe = res_df.loc[res_df['sharpe'].idxmax()]

print(f'\nOptimal by TOTAL PnL:  thr={best_total["threshold"]:.5f} ({best_total["thr_vs_spread"]:.1f}x spread) | {best_total["n_trades"]:.0f} trades | WR={best_total["win_rate"]:.1%} | PnL={best_total["total_pnl"]:.4f}')
print(f'Optimal by DAILY PnL:  thr={best_daily["threshold"]:.5f} ({best_daily["thr_vs_spread"]:.1f}x spread) | {best_daily["n_trades"]:.0f} trades | WR={best_daily["win_rate"]:.1%} | PnL={best_daily["daily_pnl"]:.6f}')
print(f'Optimal by SHARPE:     thr={best_sharpe["threshold"]:.5f} ({best_sharpe["thr_vs_spread"]:.1f}x spread) | {best_sharpe["n_trades"]:.0f} trades | WR={best_sharpe["win_rate"]:.1%} | Sharpe={best_sharpe["sharpe"]:.1f}')

---
## 2. Trades/Day vs EV Trade-off

The key chart: at what point does lowering the threshold give us more daily PnL despite lower WR?

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(DARK)
ax1.set_facecolor(DARK)

x = res_df['thr_vs_spread'].values

# Daily PnL on left axis
color1 = GOLD
ax1.plot(x, res_df['daily_pnl'].values * 10000, 'o-', color=color1, linewidth=2, markersize=5, label='Daily PnL (pips)')
ax1.set_xlabel('Threshold (x spread)', color='white', fontsize=11)
ax1.set_ylabel('Daily PnL (pips)', color=color1, fontsize=11)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.tick_params(colors='white')
ax1.axhline(0, color=RED, linewidth=0.5, linestyle='--', alpha=0.5)
ax1.axvline(1.0, color='white', linewidth=0.5, linestyle='--', alpha=0.3, label='Current (1x spread)')

# Trades/day on right axis
ax2 = ax1.twinx()
color2 = BLUE
ax2.plot(x, res_df['trades_per_day'].values, 's--', color=color2, linewidth=1.5, markersize=4, alpha=0.7, label='Trades/day')
ax2.set_ylabel('Trades / Day', color=color2, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color2)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, facecolor='#1a2332', labelcolor='white', fontsize=9)

ax1.set_title('Daily PnL vs Threshold — Finding the Sweet Spot', color='white', fontsize=13)
for spine in ax1.spines.values(): spine.set_edgecolor('#1a2332')
for spine in ax2.spines.values(): spine.set_edgecolor('#1a2332')
plt.tight_layout()
plt.show()

---
## 3. Per-Pair Threshold Analysis

Some pairs might have a lower optimal threshold — let's check each independently.

In [ ]:
pair_list = sorted(df_test['pair'].unique())
test_thresholds = [0, 0.00005, 0.00010, 0.00015, 0.00020, AVG_SPREAD, 0.00040, 0.00056]

print(f'{"Pair":<10}', end='')
for thr in test_thresholds:
    label = f'{thr/AVG_SPREAD:.1f}x'
    print(f' {label:>12}', end='')
print()
print('-' * (10 + 13 * len(test_thresholds)))

pair_best = {}

for pair in pair_list:
    pmask = pairs == pair
    print(f'{pair:<10}', end='')
    
    best_daily_pnl = -999
    best_thr = AVG_SPREAD
    
    for thr in test_thresholds:
        mask = (abs_q50 > thr) & valid & pmask
        n = mask.sum()
        if n < 5:
            print(f' {"--":>12}', end='')
            continue
        
        pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
        wr = (pred_dir[mask] == np.sign(y_test[mask])).mean()
        daily = pnl.sum() / test_days
        
        if daily > best_daily_pnl:
            best_daily_pnl = daily
            best_thr = thr
        
        print(f' {n:>3}t {wr:>4.0%} {pnl.sum():>+.3f}', end='')
    
    pair_best[pair] = best_thr
    print()

print()
print('Best threshold per pair (by daily PnL):')
for pair, thr in pair_best.items():
    print(f'  {pair}: {thr:.5f} ({thr/AVG_SPREAD:.1f}x spread)')

---
## 4. Alternative Filters — Quantile Spread (IQR)

Instead of just |Q50|, use the predicted IQR (Q75 - Q25) as an uncertainty measure.
- **Tight IQR** = model is confident about the range → trade
- **Wide IQR** = model is uncertain → skip

Combine: trade when |Q50| > some threshold AND IQR < some max.

In [ ]:
iqr = q75 - q25  # predicted interquartile range
pred_range_90 = q90 - q10  # 80% prediction interval

print('IQR (Q75-Q25) statistics:')
print(f'  Mean: {iqr[valid].mean():.6f}')
print(f'  Median: {np.median(iqr[valid]):.6f}')
print(f'  P25: {np.percentile(iqr[valid], 25):.6f}')
print(f'  P75: {np.percentile(iqr[valid], 75):.6f}')
print()

# Sweep: |Q50| threshold (rows) x IQR max (columns)
q50_thrs = [0, 0.00005, 0.00010, 0.00015, AVG_SPREAD]
iqr_maxes = [np.inf, np.percentile(iqr[valid], 75), np.percentile(iqr[valid], 50), np.percentile(iqr[valid], 25)]
iqr_labels = ['no_cap', 'P75', 'P50', 'P25']

print(f'{"":>12}', end='')
for label in iqr_labels:
    print(f' {"IQR<" + label:>18}', end='')
print()
print('-' * (12 + 19 * len(iqr_labels)))

for q50_thr in q50_thrs:
    label = f'|Q50|>{q50_thr/AVG_SPREAD:.1f}x'
    print(f'{label:>12}', end='')
    
    for iqr_max in iqr_maxes:
        mask = (abs_q50 > q50_thr) & (iqr < iqr_max) & valid
        n = mask.sum()
        if n < 10:
            print(f' {"--":>18}', end='')
            continue
        
        pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
        wr = (pred_dir[mask] == np.sign(y_test[mask])).mean()
        daily = pnl.sum() / test_days
        
        print(f' {n:>4}t {wr:>4.0%} d={daily*1e4:>+5.1f}p', end='')
    print()

---
## 5. Confidence Ratio Filter — |Q50| / IQR

A signal-to-noise ratio: how much of the predicted range is directional vs uncertain.
High |Q50|/IQR = strong directional conviction relative to uncertainty.

In [ ]:
conf_ratio = abs_q50 / (iqr + 1e-10)

print('Confidence ratio |Q50|/IQR statistics:')
print(f'  Mean: {conf_ratio[valid].mean():.3f}')
print(f'  Median: {np.median(conf_ratio[valid]):.3f}')
print(f'  P75: {np.percentile(conf_ratio[valid], 75):.3f}')
print(f'  P90: {np.percentile(conf_ratio[valid], 90):.3f}')
print()

conf_thresholds = np.arange(0, 1.5, 0.05)
conf_results = []

for thr in conf_thresholds:
    mask = (conf_ratio > thr) & valid
    n = mask.sum()
    if n < 10:
        continue
    
    pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
    wr = (pred_dir[mask] == np.sign(y_test[mask])).mean()
    
    conf_results.append({
        'threshold': thr,
        'n_trades': n,
        'trades_per_day': n / test_days,
        'win_rate': wr,
        'ev': pnl.mean(),
        'total_pnl': pnl.sum(),
        'daily_pnl': pnl.sum() / test_days,
    })

conf_df = pd.DataFrame(conf_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(DARK)

for ax, (col, title, color) in zip(axes, [
    ('trades_per_day', 'Trades/Day', BLUE),
    ('win_rate', 'Win Rate', GREEN),
    ('daily_pnl', 'Daily PnL', GOLD),
]):
    ax.set_facecolor(DARK)
    ax.plot(conf_df['threshold'], conf_df[col], 'o-', color=color, markersize=4)
    ax.set_title(title, color='white')
    ax.set_xlabel('|Q50|/IQR threshold', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    if col == 'win_rate': ax.axhline(0.5, color=RED, linewidth=0.5, linestyle='--', alpha=0.5)
    if col == 'daily_pnl': ax.axhline(0, color=RED, linewidth=0.5, linestyle='--', alpha=0.5)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Confidence Ratio Filter — |Q50| / IQR', color='white', fontsize=13)
plt.tight_layout()
plt.show()

best_conf = conf_df.loc[conf_df['daily_pnl'].idxmax()]
print(f'\nBest by daily PnL: ratio>{best_conf["threshold"]:.2f} | {best_conf["n_trades"]:.0f} trades ({best_conf["trades_per_day"]:.1f}/day) | WR={best_conf["win_rate"]:.1%} | daily={best_conf["daily_pnl"]*1e4:.1f} pips')

---
## 6. Equity Curves — Comparing Filter Strategies

Plot cumulative PnL for the best configurations found above.

In [ ]:
# Define strategies to compare
strategies = {}

# Strategy A: Current (|Q50| > 1x spread)
mask_a = (abs_q50 > AVG_SPREAD) & valid
strategies['Current (1x spread)'] = mask_a

# Strategy B: Best total PnL threshold from sweep
best_thr = best_total['threshold']
mask_b = (abs_q50 > best_thr) & valid
strategies[f'Best sweep ({best_thr/AVG_SPREAD:.1f}x)'] = mask_b

# Strategy C: No filter (trade every valid hour)
mask_c = valid.copy()
strategies['No filter'] = mask_c

# Strategy D: Best confidence ratio
best_cr = best_conf['threshold']
mask_d = (conf_ratio > best_cr) & valid
strategies[f'Conf ratio >{best_cr:.2f}'] = mask_d

# Plot equity curves
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(DARK)
ax.set_facecolor(DARK)

colors = [GOLD, GREEN, RED, BLUE]
for (name, mask), color in zip(strategies.items(), colors):
    # Build time-indexed PnL series
    pnl_series = pd.Series(0.0, index=df_test.index)
    trade_pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
    pnl_series[mask] = trade_pnl
    cum_pnl = pnl_series.cumsum()
    
    n = mask.sum()
    wr = (pred_dir[mask] == np.sign(y_test[mask])).mean()
    label = f'{name}: {n} trades, WR={wr:.0%}, PnL={pnl_series.sum():.4f}'
    ax.plot(cum_pnl.index, cum_pnl.values, color=color, linewidth=1.5, label=label)

ax.axhline(0, color='white', linewidth=0.3, alpha=0.3)
ax.set_title('Equity Curves — Filter Comparison', color='white', fontsize=13)
ax.set_ylabel('Cumulative PnL (log-return)', color='white')
ax.legend(facecolor='#1a2332', labelcolor='white', fontsize=9)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')
plt.tight_layout()
plt.show()

---
## 7. Summary & Recommendation

In [ ]:
print('=' * 70)
print('THRESHOLD OPTIMIZATION SUMMARY')
print('=' * 70)
print()
print(f'{"Strategy":<30} {"Trades":>7} {"Tr/day":>7} {"WR":>7} {"EV/trade":>10} {"DailyPnL":>10}')
print('-' * 75)

for name, mask in strategies.items():
    n = mask.sum()
    pnl = pred_dir[mask] * y_test[mask] - AVG_SPREAD
    wr = (pred_dir[mask] == np.sign(y_test[mask])).mean()
    daily = pnl.sum() / test_days
    
    print(f'{name:<30} {n:>7} {n/test_days:>7.1f} {wr:>6.1%} {pnl.mean():>10.6f} {daily*1e4:>9.1f}p')

print()
print('Note: DailyPnL in pips (x10000). All evaluated on OOS test set (post Jun 2024).')